# Dataset Comparison Visualization

This notebook provides a detailed analysis of the differences between the reference and synthetic datasets.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import librosa
import librosa.display
import numpy as np
from pathlib import Path
from IPython.display import Audio, display, Markdown, clear_output
import ipywidgets as widgets

# CONFIGURATION
reports_base = Path("../reports")
data_root = Path("../datalocal")
ref_root_default = data_root / "PC-GITA_v260210_24kHz"
mapping_path = ref_root_default / "_metadata" / "PCGITAtoPD_mapping.csv"

sns.set_theme(style="whitegrid")

In [ ]:
# Find available comparisons
available_comparisons = sorted([d.name for d in reports_base.iterdir() if d.is_dir() and d.name.startswith("comparison_")])

comp_dropdown = widgets.Dropdown(
    options=available_comparisons,
    description='Comparison:',
    disabled=False,
)

main_output = widgets.Output()

def run_visualization(comparison_name):
    with main_output:
        clear_output(wait=True)
        
        reports_dir = reports_base / comparison_name
        
        # Load Data
        integrity_df = pd.read_csv(reports_dir / "comparison_integrity.csv")
        audio_df = pd.read_csv(reports_dir / "comparison_audio_durations.csv")
        
        # Load Metadata for H/Y
        mapping = pd.read_csv(mapping_path, sep=';')
        mapping = mapping[['Code BD-Parkinson', 'H/Y']]
        mapping.columns = ['speaker_id', 'HY']
        
        missing = integrity_df[~integrity_df['exists_in_test']]
        missing_csv = integrity_df[integrity_df['csv_exists_in_test'] == False]
        mismatches = integrity_df[integrity_df['transcription_match'] == False]
        sig_deltas = audio_df[audio_df['significant_diff'] == True]

        display(Markdown(f"# Analysis: {comparison_name}"))
        
        display(Markdown("## 1. Data Integrity & Problem Summary"))
        display(Markdown(f"""
- **Missing Audio Files**: {len(missing)}
- **Missing Alignment CSVs**: {len(missing_csv)}
- **Transcription Mismatches**: {len(mismatches)}
- **Significant Duration Deltas**: {len(sig_deltas)}
"""))

        if not missing.empty:
            display(Markdown("**Missing Audio (Top 5):**"))
            display(missing[['task', 'filename']].head(5))

        if not missing_csv.empty:
            display(Markdown("**Missing Alignment CSVs (Top 5):**"))
            display(missing_csv[['task', 'filename']].head(5))

        if not mismatches.empty:
            display(Markdown("**Transcription Mismatches:**"))
            display(mismatches[['task', 'filename']])

        if not sig_deltas.empty:
            display(Markdown("**Significant Duration Deltas (Top 5):**"))
            display(sig_deltas[['task', 'filename', 'duration_delta']].head(5))

        # --- Interactive Section ---
        display(Markdown("## 2. Interactive Pair Comparison"))
        problematic_files = sig_deltas['filename'].tolist() if not sig_deltas.empty else []
        
        if problematic_files:
            file_select = widgets.Dropdown(options=problematic_files, description='Select File:')
            pair_output = widgets.Output()
            
            def on_file_select(file_name):
                with pair_output:
                    clear_output(wait=True)
                    row = audio_df[audio_df['filename'] == file_name].iloc[0]
                    task = row['task']
                    r_path = ref_root_default / task / file_name
                    test_name = comparison_name.replace("comparison_", "").replace(ref_root_default.name + "_", "")
                    t_path = data_root / test_name / task / file_name
                    if not t_path.exists(): t_path = r_path
                    
                    if r_path.exists():
                        ref_txt = r_path.with_suffix('.txt').read_text() if r_path.with_suffix('.txt').exists() else "N/A"
                        test_txt = t_path.with_suffix('.txt').read_text() if t_path.with_suffix('.txt').exists() else "N/A"
                        display(Markdown(f"**Ref Transcript:** {ref_txt}"))
                        display(Markdown(f"**Test Transcript:** {test_txt}"))
                        
                        fig, ax = plt.subplots(2, 1, figsize=(10, 5))
                        for i, (p, label) in enumerate([(r_path, "Reference"), (t_path, "Test")]):
                            y, sr = librosa.load(p)
                            D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
                            librosa.display.specshow(D, y_axis='log', x_axis='time', ax=ax[i], sr=sr)
                            ax[i].set_title(f"{label} Spectrogram")
                            display(Audio(p))
                        plt.tight_layout()
                        plt.show()
            
            widgets.interactive_output(on_file_select, {'file_name': file_select})
            display(file_select, pair_output)
        else:
            display(Markdown("*No significant deltas to compare.* "))

        # --- Visualizations ---
        display(Markdown("## 3. Embedding Distance Analysis"))
        emb_path = reports_dir / "speaker_embedding_summary.csv"
        if emb_path.exists():
            e_df = pd.read_csv(emb_path)
            e_df = e_df.merge(mapping, on='speaker_id', how='left')
            
            # Detailed Table
            display(Markdown("### Average Distance per Speaker and Task"))
            pivot_df = e_df.pivot(index=['speaker_id', 'status', 'HY'], columns='task', values='avg_distance')
            pivot_df['AVG (all)'] = pivot_df.mean(axis=1)
            cols_no_ddk = [c for c in pivot_df.columns if c not in ['ddk', 'AVG (all)']]
            pivot_df['AVG (no ddk)'] = pivot_df[cols_no_ddk].mean(axis=1)
            
            display(pivot_df.style.background_gradient(cmap='YlOrRd', axis=None).format("{:.4f}"))
            
            display(Markdown("### Variance per Speaker and Task"))
            pivot_std = e_df.pivot(index=['speaker_id', 'status', 'HY'], columns='task', values='std_distance')
            display(pivot_std.style.background_gradient(cmap='Blues', axis=None).format("{:.4f}"))
            
            plt.figure(figsize=(10, 4))
            sns.boxplot(data=e_df, x="task", y="avg_distance", hue="status")
            plt.title("Avg Embedding Distance per Task")
            plt.xticks(rotation=45)
            plt.show()
            
        display(Markdown("## 4. Phoneme Duration Deltas"))
        ph_path = reports_dir / "comparison_phonemes.csv"
        if ph_path.exists():
            p_df = pd.read_csv(ph_path).dropna(subset=["duration_delta"])
            if not p_df.empty:
                plt.figure(figsize=(10, 4))
                sns.violinplot(data=p_df, x="task", y="duration_delta", hue="status", split=True)
                plt.title("Phoneme Duration Deltas")
                plt.xticks(rotation=45)
                plt.show()

def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        run_visualization(change['new'])

comp_dropdown.observe(on_change)
display(comp_dropdown, main_output)

# Trigger first load
if available_comparisons:
    run_visualization(available_comparisons[0])